# Agentic Workflow2

## LangGraph 워크플로우 구현 단계별 정리

이번 구현은 Agentic RAG 흐름을 `StateGraph`로 명시적으로 제어하는 예제이다. 핵심은 LLM이 바로 답을 생성하는 구조가 아니라, 현재 상태를 기준으로 `생성/도구 호출 -> 도구 실행 -> 답변 검증 -> 재시도 또는 종료` 흐름을 그래프 형태로 연결했다는 점이다.

### 1. 상태 스키마 정의

`AgenticWorkflowState`는 그래프 전체에서 공유되는 상태 저장소 역할을 한다. 질문, 검색된 문서 컨텍스트, 최종 답변, 도구 호출 정보, 검증 상태, 생성 시도 횟수를 하나의 상태로 관리한다.

- `question`: 사용자의 질문
- `context`: 검색 도구로 가져온 문서 리스트
- `answer`: LLM이 생성한 답변
- `tool_calls`: LLM이 호출하기로 결정한 도구 정보
- `verification_status`: 다음 라우팅을 결정하는 상태값
- `generation_attempts`: 답변 생성 및 재시도 횟수

특히 `verification_status`는 단순한 로그가 아니라 그래프의 다음 이동 방향을 결정하는 라우팅 키로 사용된다.

### 2. 검색 도구와 LLM 준비

Chroma 벡터 저장소와 retriever를 만들고, 이를 `retrieve_documents`라는 LangChain tool로 감싼다. 이후 `llm.bind_tools(tools)`를 통해 LLM이 필요할 때 검색 도구를 호출할 수 있게 한다.

이 구조에서 LLM은 두 가지 역할을 한다.

1. 컨텍스트가 없을 때는 도구 호출이 필요한지 판단한다.
2. 컨텍스트가 생긴 뒤에는 해당 컨텍스트를 바탕으로 답변을 생성한다.

### 3. `generate_or_call_tool` 노드

`generate_or_call_tool`은 그래프의 첫 번째 핵심 노드이다. 현재 상태에 `context`가 있는지에 따라 동작이 갈린다.

- `context`가 없으면 LLM에게 질문을 전달하고, 도구 호출 여부를 판단하게 한다.
- LLM이 도구 호출을 반환하면 `tool_calls`를 상태에 저장하고 `verification_status`를 `tool_call`로 설정한다.
- `context`가 있으면 검색 결과를 바탕으로 최종 답변을 생성하고 `verification_status`를 `verification_ready`로 설정한다.

즉, 이 노드는 Reasoning 단계와 Generation 단계를 함께 담당한다.

### 4. `execute_tool` 노드

`execute_tool`은 LLM이 요청한 도구를 실제로 실행하는 노드이다. `tool_calls`에 저장된 도구 이름과 인자를 읽고, `TOOL_MAP`에서 해당 도구를 찾아 `.invoke()`로 실행한다.

도구 실행 결과는 다시 `Document` 객체로 감싸서 `context`에 저장한다. 이렇게 해야 다음 `generate` 노드가 검색 결과를 RAG 컨텍스트처럼 사용할 수 있다.

이 노드가 끝나면 그래프는 다시 `generate` 노드로 돌아간다. 흐름은 `질문 -> 도구 호출 결정 -> 도구 실행 -> 검색 결과 기반 답변 생성`이 된다.

### 5. `verification_router` 노드

`verification_router`는 생성된 답변을 검증하고 다음 흐름을 결정한다.

- 답변에 `FAIL_INSUFFICIENT_CONTEXT`가 있으면 실패로 판단하고 재시도한다.
- 생성 시도 횟수가 3회 이상이면 `end`로 종료한다.
- 답변이 검증 조건을 만족하면 `pass`로 종료한다.
- 조건을 만족하지 못하면 `fail`로 설정하고 다시 `generate`로 보낸다.

여기서 중요한 점은 검증 함수가 직접 다음 노드를 호출하지 않는다는 것이다. 함수는 상태값만 바꾸고, 실제 이동은 LangGraph의 조건부 엣지가 담당한다.

### 6. 그래프 구성

`StateGraph(AgenticWorkflowState)`로 그래프 객체를 만들고 세 개의 노드를 등록한다.

```python
workflow.add_node("generate", generate_or_call_tool)
workflow.add_node("execute_tool", execute_tool)
workflow.add_node("verify_step", verification_router)
```

시작점은 `generate`로 설정한다.

```python
workflow.set_entry_point("generate")
```

이후 조건부 엣지를 통해 상태값에 따라 다음 노드를 선택한다.

- `generate` 이후 `tool_call`이면 `execute_tool`로 이동
- `generate` 이후 답변 생성이 끝났으면 `verify_step`으로 이동
- `execute_tool` 이후에는 다시 `generate`로 이동
- `verify_step` 이후 `pass` 또는 `end`이면 종료
- `verify_step` 이후 `fail`이면 다시 `generate`로 이동

전체 흐름은 다음과 같다.

```text
generate
  -> execute_tool -> generate
  -> verify_step
       -> pass/end -> END
       -> fail -> generate
```

### 7. 컴파일과 실행

마지막으로 `workflow.compile()`을 호출해 실행 가능한 LangGraph 앱으로 변환한다. 이후 `app.invoke(initial_state)`에 초기 상태를 넣으면 그래프가 상태를 갱신하면서 정의된 흐름대로 실행된다.

초기 상태에는 질문과 빈 컨텍스트, 빈 답변, 시도 횟수 0, 빈 도구 호출 목록을 넣는다. 그래프는 이 상태를 출발점으로 삼아 필요한 경우 검색 도구를 호출하고, 검색 결과를 바탕으로 답변을 생성한 뒤 검증까지 수행한다.

### Workflow example

import os
from typing import TypedDict, List, Annotated
from operator import add
from langchain_core.documents import Document
from langchain_core.runnables import Runnable
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY" # 실제 키 설정 필요

class AgenticWorkflowState(TypedDict):
    """
    Agentic Workflow의 현재 상태를 나타내는 TypedDict.
    """
    question: str  # 사용자 질문
    context: List[Document]  # 검색된 문서 컨텍스트
    answer: str  # LLM이 생성한 최종 답변
    tool_calls: List[dict] # LLM이 결정한 도구 호출 정보 (raw JSON/dict)
    verification_status: str # 현재 상태/다음 라우팅 키: "tool_call", "verification_ready", "pass", "fail", "end"
    generation_attempts: Annotated[int, add] # 답변 생성 시도 횟수 누적 # 그래프 안에서 얼만큼 시도했냐? 라는 인자
    
    
# ChromaDB 및 Retriever 셋업 (더미 데이터)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = Chroma.from_texts(
    ["LangGraph는 복잡한 에이전트 워크플로우를 상태 기반으로 구현한다.",
     "Agentic RAG는 LLM의 추론과 도구 사용을 결합하여 정확도를 높인다.",
     "LangChain Expression Language (LCEL)은 Runnable 체인을 구성하는 기본 방식이다."],
    embeddings
)
retriever_runnable: Runnable = db.as_retriever(search_kwargs={"k": 3}) #: Runnable은 타입 힌트이다. 즉, 파이썬에게 retriever_runnable은 Runnable으로 보아라!
# Runnable은 랭체인에서 invoke(), stream()과 같은 공통 실행 인터페이스를  가진 객체이다. 

# LLM과 Tools 정의
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

@tool # tool 데코레이터를 상속받아서 tool을 만들었음.  create_retriever_tool를 써도 된다. 하지만, @tool 방식은 직접 커스텀 가능
def retrieve_documents(query: str) -> str:
    """
    벡터 저장소에서 질문과 관련된 문서를 검색합니다. 이 도구는 문자열로 된 검색 결과를 반환합니다.
    LLM은 이 도구를 호출하기 전에, 검색 쿼리를 최적화하여 입력해야 합니다.
    """
    print(f"\n[Tool] 🔍 검색 도구 실행 (쿼리: {query})")
    docs = retriever_runnable.invoke(query)
    context_str = "\n\n".join([f"[Source {i+1}] {doc.page_content}" for i, doc in enumerate(docs)])
    return context_str

tools = [retrieve_documents]
llm_with_tools = llm.bind_tools(tools)

# 도구 이름과 실제 함수 매핑 (실행을 위해 필요)
TOOL_MAP = {t.name: t for t in tools}

In [ ]:
def generate_or_call_tool(state: AgenticWorkflowState) -> AgenticWorkflowState: # 노드임. 나중에 등록할 예정 (add_node 방식으로)
    """LLM이 답변을 생성하거나, 도구 호출을 결정합니다."""
    print("--- 🧠 1. 추론/결정 노드 실행 ---")
    question = state["question"]
    context = state["context"]
    
    # 1. 컨텍스트가 있다면, 최종 답변 생성 (Generation)
    if context:
        context_str = "\n\n".join([doc.page_content for doc in context])
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", 
                 "당신은 Agentic Workflow의 최종 생성 모듈입니다. 제공된 Context를 기반으로 질문에 간결하고 전문적인 한국어로 답변하십시오. "
                 "Context가 부족하여 답할 수 없으면, 명시적으로 **'FAIL_INSUFFICIENT_CONTEXT'**라는 키워드만 반환하십시오."
                 ),
                ("human", "Context:\n{context}\n\nQuestion: {question}"),
            ]
        )
        response: AIMessage = llm.invoke(prompt.format_messages(context=context_str, question=question))
        
        # 답변과 검증 준비 상태 반환
        # generation_attempts는 도구 호출 시에만 증가시키므로 여기서는 0으로 설정
        return {"answer": response.content, "verification_status": "verification_ready", "generation_attempts": 0} 

    # 2. 컨텍스트가 없다면 (초기 진입), 도구 사용 결정 (Action)
    else:
        # LLM에게 도구 사용을 유도 (함수 호출)
        response: AIMessage = llm_with_tools.invoke([HumanMessage(content=question)])
        
        if response.tool_calls:
            print("   -> [Action] 도구 호출을 결정했습니다.")
            # tool_calls 정보를 상태에 저장하고, 시도 횟수를 1 증가
            return {"tool_calls": response.tool_calls, "verification_status": "tool_call", "generation_attempts": 1}
        else:
             # 도구 호출 없이 바로 답변 시 (매우 드문 경우)
             return {"answer": response.content, "verification_status": "verification_ready", "generation_attempts": 1}

#### 2-2. 노드 2: 도구 실행 (`execute_tool`)

# 이 노드는 LLM이 요청한 **행동(Action)**을 실제 환경에서 실행하고 그 결과를 **관찰(Observation)**로 반환합니다.

def execute_tool(state: AgenticWorkflowState) -> AgenticWorkflowState:
    """LLM이 요청한 도구를 직접 실행하고 결과를 반환합니다."""
    print("--- 🛠️ 2. 도구 실행 노드 실행 ---")
    tool_calls = state["tool_calls"]
    retrieved_context_docs = []
    
    for call in tool_calls:
        tool_name = call['name']
        tool_args = call['args']
        
        if tool_name in TOOL_MAP:
            tool_function = TOOL_MAP[tool_name]
            
            # **전문적 구현: LangChain 최신 표준인 .invoke()를 사용하여 도구 실행**
            tool_output_str: str = tool_function.invoke(tool_args) 
            
            # 검색 결과를 Document 객체로 변환 (다음 RAG 단계에 맞춤)
            retrieved_context_docs.append(Document(page_content=tool_output_str)) # 앞서 context에 Document 객체 리스트가 들어오도록 설계했다. Document 객체여야지 .page_content를 가진다. 
        else:
            print(f"경고: 알 수 없는 도구 {tool_name} 호출")

    # 상태 업데이트: 검색된 컨텍스트를 추가하고 generate 노드로 복귀 준비
    return {"context": retrieved_context_docs, "verification_status": "generation_ready"}


# 참고: Document 객체는 랭체인에서 문서 한 조각을 표현하는 표준 객체이다. page_content와 metadata를 가진다. 
# 랭체인에서 Document 객체를 쓰는 이유는 텍스트 내용과 출저 정보를 함께 관리할 수 있고(나중에 source 표시가 쉬움) retriver가 보통 Documen 리스트를 반환하며, 벡터스토에에 저장하기 쉽기 때문이다

In [ ]:
# 라우팅 처리해주는 함수 
# LLM이 만든 답변을 검사하고, 다음 그래프가 어디로 갈지 결정하는 함수이다. 즉, 답변 검증 + 다음 라우팅 상태 결정 

def verification_router(state: AgenticWorkflowState) -> AgenticWorkflowState:
    """답변 검증 로직을 수행하고 다음 라우팅 키("pass", "fail", "end")를 상태에 저장합니다."""
    print("--- 🧐 3. 검증/자율 수정 노드 실행 ---")
    answer = state["answer"]  # 답변 상태에 따라 verification_status를 바꾼다. 
    
    # 1. LLM이 'FAIL_INSUFFICIENT_CONTEXT'를 반환했는지 확인 (LLM의 자율적인 판단)
    if "FAIL_INSUFFICIENT_CONTEXT" in answer:
        print("   -> [Reasoning] LLM이 컨텍스트 부족을 판단하여 재시도를 요청함.")
        return {"verification_status": "fail"}
    
    # 2. 시도 횟수 제한 검사
    if state["generation_attempts"] >= 3:
        print("   -> [End] 최대 시도 횟수(3회) 초과. 종료합니다.")
        return {"verification_status": "end"} 

    # 3. 최종 검증 (외부 로직 - 예시) -> 즉, 최종 답변으로 인정
    if "LangGraph" in answer and ("상태 기반" in answer or "워크플로우" in answer):
        print("   -> [Reasoning] 검증 통과.")
        return {"verification_status": "pass"}
    
    # 4. 실패 및 재시도
    print("   -> [Reasoning] 검증 실패. 재시도를 유도합니다.")
    # 실패 시 generation_attempts에 1을 추가하여 카운트를 올리고 fail 상태 반환
    return {"verification_status": "fail", "generation_attempts": 1}

# 결론적으로 이 함수를 거치면 pass fail end중 하나를 state에 기록하게 된다. -> 즉, 답변을 검증하고 그래프의 다음 이동 방향을 결정하는 라우터 역할을 한다. 
# 이 함수는 상태값만 바꾼다. 이후 조건부 엣지를 설정해서 이 값을 읽고 다음 노드로의 이동을 결정할 것이다. (여기서 기록된 상태에 따라 노드 이동을 결정한다)

In [ ]:
# --- 3. LangGraph 워크플로우 빌드 ---

# 1. StateGraph 객체 생성
workflow = StateGraph(AgenticWorkflowState)

# 2. 노드 추가
workflow.add_node("generate", generate_or_call_tool)
workflow.add_node("execute_tool", execute_tool)
workflow.add_node("verify_step", verification_router) 

# 3. 시작점 정의
workflow.set_entry_point("generate") #set_entry_point 를 통해서 generate라는 함수를 시작점으로 정해준다. 

# 4. 엣지 정의 및 라우팅

# 4-1. generate 노드 후 라우팅 
# generate함수로부터 어디로 갈지 정의를 해준다. 
def route_after_generation(state: AgenticWorkflowState) -> str:
    """Generate 노드 후 라우팅: 도구 호출 vs. 검증 노드 호출"""
    if state.get("verification_status") == "tool_call":
        return "execute_tool" # Action: 도구 호출 필요
    return "verify_step" # Generation 완료: 검증 필요

workflow.add_conditional_edges(
    "generate",
    route_after_generation,
    {
        "execute_tool": "execute_tool",
        "verify_step": "verify_step",
    }
)

# 4-2. execute_tool 노드 후에는 generate 노드로 복귀 (검색 결과를 바탕으로 답변 생성 시도)
workflow.add_edge("execute_tool", "generate")

# 4-3. 검증 노드 후의 최종 라우팅
workflow.add_conditional_edges(
    "verify_step",
    lambda state: state["verification_status"], # 상태의 verification_status 값을 라우팅 키로 사용
    {
        "pass": END,  # 통과 시 종료
        "fail": "generate",  # 실패 시 generate 노드로 복귀 (재시도)
        "end": END,   # 최대 시도 횟수 초과 시 종료
    }
)

# 5. Graph 컴파일
app = workflow.compile()
print("\n✅ Agentic Workflow LangGraph 워크플로우 구성 완료 및 컴파일됨.")


In [ ]:
# --- 4. 최종 실행 ---
initial_state = {
    "question": "복잡한 에이전트 워크플로우를 구현하는 데 효과적인 기술이 뭐야?.", 
    "context": [], 
    "answer": "", 
    "generation_attempts": 0,
    "tool_calls": [],
    "verification_status": ""
}

final_state = app.invoke(initial_state)

print("\n" + "="*80)
print("--- 🌟 Agentic Workflow 최종 실행 결과 ---")
print(f"총 시도 횟수: {final_state['generation_attempts']}회")
print(f"검증 상태: {final_state['verification_status']}")
print(f"질문: {final_state['question']}")
print("-" * 20)
print(f"답변:\n{final_state['answer']}")
print("="*80)

# 이렇게 워크플로우를 만들면, 질문 자체를 하나의 question에 넣는 것이 아닌, state에 정의된, initial_state에 정의된 스키마 형태로 question을 넣어주게 되고, 이러한 질문에 대해서 답변이 나오게 된다. 

예상답변

```python
--- 🧠 1. 추론/결정 노드 실행 ---
   -> [Action] 도구 호출을 결정했습니다.
--- 🛠️ 2. 도구 실행 노드 실행 ---

[Tool] 🔍 검색 도구 실행 (쿼리: 복잡한 에이전트 워크플로우 구현 기술)
--- 🧠 1. 추론/결정 노드 실행 ---
--- 🧐 3. 검증/자율 수정 노드 실행 ---
   -> [Reasoning] 검증 통과.

================================================================================
--- 🌟 Agentic Workflow 최종 실행 결과 ---
총 시도 횟수: 1회
검증 상태: pass
질문: 복잡한 에이전트 워크플로우를 구현하는 데 효과적인 기술이 뭐야?.
--------------------
답변:
LangGraph는 복잡한 에이전트 워크플로우를 구현하는 데 효과적인 기술입니다.
================================================================================
```

코드가 복잡해 보이지만, 중요한 것은

결국 state 그래프를 정의하고 노드 추가하고 시작점 정의하고 edge를 add edage로 연결하거나 라우팅이 있는 edage로 연결할 수있고.. 이를 컴파일 한 후에 스키마로 정의된 question을 컴파일 그래프에 넣어서 답변을 얻는 구조이다. 

이런 구조가 랭체인 에이전트보다 좋은 점은 제어가 가능하다는 것이다!! 노드 추가 라우팅하는 등등.. 

내가 만들고자 하는 에이전트가 하나하나 스텝별로 복잡하다고 하다면, 랭그래프 기반으로 워크플로우를 구성하는 것이 랭체인 기반 에이전트보다는 더 자율성 있게 커스텀하는 에이전트를 만들 수 있을 것이다. 


## 핵심 요약

LangGraph 구현의 핵심은 에이전트의 행동을 노드와 엣지로 명시적으로 분리하는 것이다. 일반적인 LLM 체인에서는 모델 출력에 많은 흐름을 맡기지만, LangGraph에서는 `상태`, `노드`, `조건부 라우팅`, `종료 조건`을 코드로 직접 정의할 수 있다.

이 예제에서는 `generate`가 판단과 생성을 담당하고, `execute_tool`이 외부 도구 실행을 담당하며, `verify_step`이 답변 검증과 재시도 여부를 담당한다. 각 노드는 상태를 업데이트하고, 조건부 엣지는 업데이트된 상태를 읽어 다음 이동 경로를 결정한다.

따라서 LangGraph는 복잡한 Agentic RAG처럼 도구 사용, 반복, 검증, 종료 조건이 필요한 워크플로우를 만들 때 유용하다. 특히 각 단계를 직접 제어할 수 있기 때문에 단순한 랭체인 에이전트보다 커스텀 가능한 에이전트 구조를 만들기 좋다.